In [ ]:
from sklearn.linear_model import LassoCV
from helper_functions import LSE, construct_R, decomp_orthog
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt  

In [ ]:
#Specify parameters for experiment
a = [1,5,25]#[1,5,25] # 1, 5 and 25 are options
decomp_lvl = [2,3,6] # 2, 3, and 6 are options
N = [200,400,800] #number of samples
outer_rep = 20
var_error = 1.0 #variance of error terms.

In [ ]:
#This function, given an NxI np-array Mat, creates a new NXM np array Mat_twoway which contains all main-effects, quadratic terms, and
#two-way interactions of the columns in Mat
def two_way_interaction(Mat):

    #Retrieve the number of columns in Mat
    num_cols_Mat = np.shape(Mat)[1]
    
    Mat_twoway = np.concatenate((Mat,Mat**2),axis = 1)
    for i in range(num_cols_Mat):
        for j in range(i+1, num_cols_Mat):
            interact_ij = np.array([Mat[:,i]*Mat[:,j]]).T
            Mat_twoway = np.concatenate((Mat_twoway,interact_ij), axis = 1)

    return Mat_twoway

In [ ]:
#This function is used for creating an MxJ np array of true coefficients for each of the nine settings.
#Recall we assume there are 12 decision variables and 6 objectives.
#We also create an MxJ np array of perturbed coefficients for each of the nine settings.
def true_coefficient_matrix(decomp,alpha,dense_sparse,epsilon):
    #decomp: This corresponds to list decomp_lvl. We expect decomp_lvl = 2,3,6
    #alpha: This corresponds to number a. We expect a = 0.5,1,2
    #dense_sparse: 0 - dense, 1 - sparse, -1 - no perturbed coefficients
    #epsilon: This corresponds to a number epsilon. Should be relatively small.

    if decomp == 2:
        clust_1 = np.array([[1,1,1,1,1,1,0,0,0,0,0,0]])
        clust_2 = np.array([[0,0,0,0,0,0,1,1,1,1,1,1]])
        two_way_clust_1 = two_way_interaction(clust_1)
        two_way_clust_2 = two_way_interaction(clust_2)
        true_coefficient = alpha*np.array([two_way_clust_1[0],two_way_clust_1[0],two_way_clust_1[0],two_way_clust_2[0],two_way_clust_2[0],two_way_clust_2[0]]).T

        if dense_sparse == 0:
            pert_1 = np.array([[0,0,0,0,0,0,1,1,1,1,1,1]])
            pert_2 = np.array([[1,1,1,1,1,1,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_2[0]]).T
        elif dense_sparse == 1:
            pert_1 = np.array([[0,0,0,0,0,0,1,1,1,0,0,0]])
            pert_2 = np.array([[1,1,1,0,0,0,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_2[0]]).T
        elif dense_sparse == -1:
            pert_1 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_2 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            pert_coefficient = np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_2[0]]).T
                
    elif decomp == 3:
        clust_1 = np.array([[1,1,1,1,0,0,0,0,0,0,0,0]])
        clust_2 = np.array([[0,0,0,0,1,1,1,1,0,0,0,0]])
        clust_3 = np.array([[0,0,0,0,0,0,0,0,1,1,1,1]])
        two_way_clust_1 = two_way_interaction(clust_1)
        two_way_clust_2 = two_way_interaction(clust_2)
        two_way_clust_3 = two_way_interaction(clust_3)
        true_coefficient = alpha*np.array([two_way_clust_1[0],two_way_clust_1[0],two_way_clust_2[0],two_way_clust_2[0],two_way_clust_3[0],two_way_clust_3[0]]).T

        if dense_sparse == 0:
            pert_1 = np.array([[0,0,0,0,1,1,1,1,1,1,1,1]])
            pert_2 = np.array([[1,1,1,1,0,0,0,0,1,1,1,1]])
            pert_3 = np.array([[1,1,1,1,1,1,1,1,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_3[0]]).T
        elif dense_sparse == 1:
            pert_1 = np.array([[0,0,0,0,1,1,0,0,1,1,0,0]])
            pert_2 = np.array([[1,1,0,0,0,0,0,0,1,1,0,0]])
            pert_3 = np.array([[1,1,0,0,1,1,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_3[0]]).T
        elif dense_sparse == -1:
            pert_1 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_2 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_3 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            pert_coefficient = np.array([two_way_pert_1[0],two_way_pert_1[0],two_way_pert_2[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_3[0]]).T

    elif decomp == 6:
        clust_1 = np.array([[1,1,0,0,0,0,0,0,0,0,0,0]])
        clust_2 = np.array([[0,0,1,1,0,0,0,0,0,0,0,0]])
        clust_3 = np.array([[0,0,0,0,1,1,0,0,0,0,0,0]])
        clust_4 = np.array([[0,0,0,0,0,0,1,1,0,0,0,0]])
        clust_5 = np.array([[0,0,0,0,0,0,0,0,1,1,0,0]])
        clust_6 = np.array([[0,0,0,0,0,0,0,0,0,0,1,1]])
        two_way_clust_1 = two_way_interaction(clust_1)
        two_way_clust_2 = two_way_interaction(clust_2)
        two_way_clust_3 = two_way_interaction(clust_3)
        two_way_clust_4 = two_way_interaction(clust_4)
        two_way_clust_5 = two_way_interaction(clust_5)
        two_way_clust_6 = two_way_interaction(clust_6)
        true_coefficient = alpha*np.array([two_way_clust_1[0],two_way_clust_2[0],two_way_clust_3[0],two_way_clust_4[0],two_way_clust_5[0],two_way_clust_6[0]]).T

        if dense_sparse == 0:
            pert_1 = np.array([[0,0,1,1,1,1,1,1,1,1,1,1]])
            pert_2 = np.array([[1,1,0,0,1,1,1,1,1,1,1,1]])
            pert_3 = np.array([[1,1,1,1,0,0,1,1,1,1,1,1]])
            pert_4 = np.array([[1,1,1,1,1,1,0,0,1,1,1,1]])
            pert_5 = np.array([[1,1,1,1,1,1,1,1,0,0,1,1]])
            pert_6 = np.array([[1,1,1,1,1,1,1,1,1,1,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            two_way_pert_4 = two_way_interaction(pert_4)
            two_way_pert_5 = two_way_interaction(pert_5)
            two_way_pert_6 = two_way_interaction(pert_6)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_4[0],two_way_pert_5[0],two_way_pert_6[0]]).T
        elif dense_sparse == 1:
            pert_1 = np.array([[0,0,1,0,1,0,1,0,1,0,1,0]])
            pert_2 = np.array([[1,0,0,0,1,0,1,0,1,0,1,0]])
            pert_3 = np.array([[1,0,1,0,0,0,1,0,1,0,1,0]])
            pert_4 = np.array([[1,0,1,0,1,0,0,0,1,0,1,0]])
            pert_5 = np.array([[1,0,1,0,1,0,1,0,0,0,1,0]])
            pert_6 = np.array([[1,0,1,0,1,0,1,0,1,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            two_way_pert_4 = two_way_interaction(pert_4)
            two_way_pert_5 = two_way_interaction(pert_5)
            two_way_pert_6 = two_way_interaction(pert_6)
            pert_coefficient = epsilon*np.array([two_way_pert_1[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_4[0],two_way_pert_5[0],two_way_pert_6[0]]).T
        elif dense_sparse == -1:
            pert_1 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_2 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_3 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_4 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_5 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            pert_6 = np.array([[0,0,0,0,0,0,0,0,0,0,0,0]])
            two_way_pert_1 = two_way_interaction(pert_1)
            two_way_pert_2 = two_way_interaction(pert_2)
            two_way_pert_3 = two_way_interaction(pert_3)
            two_way_pert_4 = two_way_interaction(pert_4)
            two_way_pert_5 = two_way_interaction(pert_5)
            two_way_pert_6 = two_way_interaction(pert_6)
            pert_coefficient = np.array([two_way_pert_1[0],two_way_pert_2[0],two_way_pert_3[0],two_way_pert_4[0],two_way_pert_5[0],two_way_pert_6[0]]).T

    return true_coefficient,pert_coefficient

In [ ]:
bluh,blah = true_coefficient_matrix(2,1,1,0.1)
print(bluh)
rng = np.random.default_rng(1000)
print(two_way_interaction(rng.uniform(0.0,1.0,size = (20,3))))

In [ ]:
#Given a matrix of fitted parameters (an MxJ np array), create a corresponding indicator matrix (MxJ np array) where each 
#entry is 1 if the parameter is larger than some threshold delta, and 0 otherwise.
def active_indicator(coefficient_matrix,delta = 0.000001):
    #coefficient_matrix: A matrix of fitted coefficients
    #delta: A parameter which decides the cutoff between a term being active and inactive.

    M,J = np.shape(coefficient_matrix)
    
    indicator = np.zeros((M,J))

    for j in range(J):
        for m in range(M):
            if abs(coefficient_matrix[m,j]) >= delta:
                indicator[m,j] = 1
            else:
                indicator[m,j] = 0

    return indicator

In [ ]:
#This function calculates the mean-squared-error between the estimated coefficient matrix and true_coefficient_matrix.
def global_mse(est_coefficient_matrix,true_coefficient_matrix):
    #est_coefficient_matrix: A MxJ np array of estimated coefficients.
    #true_coefficient_matrix: A MxJ np array of the true coefficients.
    
    return np.mean((true_coefficient_matrix - est_coefficient_matrix)**2)

In [ ]:
#This function is used for checking if two coefficient matrices have the same active variables (0 means no active variables in common,
#1 means all variables are in common). The coefficient matrices
#should first be converted to indicator matrices by using active_indicator.
def check_variable_selection(est_indicator,true_indicator):
    #est_indicator: A MxJ np array denoting the active variables in the estimated model.
    #true_indicator: A MxJ np array denoting the active variables in the true model.
    return np.mean(est_indicator == true_indicator)

In [ ]:
#This function returns an NxJ np array of simulated responses. 
def response_simulator(model_matrix,coefficients,error):
    #model_matrix: This is an NxM np array which has information on the model terms. Typically in this experiment,
                   #The first I columns are main effects, the next I columns are
                   #quadratics, and the remaining terms are interaction terms.
    #coefficients: This is a MxJ np array which determines the effect of each model term. J are the number of objective functions.
    #error: This is an NxJ np array of error terms. In this experiment they will be drawn independently from a standard normal distribution.

    responses = model_matrix@coefficients + error

    return responses

In [ ]:
#Constructs a list to be used in decomp_orthog to indicate heredity relations. We assume a full quadratics-interactions effects model is being fit. We assume that
#there is no constant term in the model.
def heredity_list(I,J):
    #I - number of input variables
    #J - number of objective function
    
    inner_list = []

    #Main effects
    for i in range(I):
        inner_list.append([])

    #Quadratics
    for i in range(I):
        inner_list.append([i])

    #Interactions
    for i in range(I):
        for i_2 in range(i+1,I):
            inner_list.append([i,i_2])

    D_her = [inner_list for j in range(J)]

    return D_her

In [ ]:
#This function takes the scaling factors from decomp_orthog and multiplies them by the LSE estimates to get the scaled coefficients.
def decomp_scaled_coefficients(theta,lse_coeff):
    #Theta: this should be an MxJ numpy array of scaling factors obtained from decomp_orthog.
    #lse_coeff: this should be an MxJ numpy array of estimated coefficients coming from LSE, and should be the LSE estimates used in constructing matrices R_1,...
    #R_J for decomp_orthog.

    return theta*lse_coeff

In [ ]:
#This function fits J Lasso regression and gathers the estimated coefficients into an MxJ numpy array.
def gather_lasso_reg(model_matrix,responses):
    #model_matrix: This is an NxM np array which has information on the model terms. Typically in this experiment,
                   #The first I columns are main effects, the next I columns are
                   #quadratics, and the remaining terms are interaction terms.
    #responses: This is an NxJ np array containing N responses for each of the J objective functions. Should be constructed using response_simulator.

    J = len(responses[0])
    
    lasso_coeff_matrix = []
    for j in range(J):
        lasso_j = LassoCV(fit_intercept = False).fit(model_matrix,responses[:,j])
        lasso_coeff_matrix.append(lasso_j.coef_)

    return np.array(lasso_coeff_matrix).T

In [ ]:
#Test/try out LassoCV and decomp_orthog
#rng_test = np.random.default_rng(1000)
#test_N = 800
#test_inputs = rng_test.uniform(0.0,1.0,size = (test_N,12))
#test_errors = rng_test.normal(0.0,1.0,size = (test_N,6))

#test_decomp_lvl_curr = 6
#test_a_curr = 25
#test_true_coeff = true_coefficient_matrix(test_decomp_lvl_curr,test_a_curr)

#test_model = two_way_interaction(test_inputs)

#test_responses = response_simulator(test_model,test_true_coeff,test_errors)

#print(gather_lasso_reg(test_model,test_responses))

#print('true_coeff: '+ str(test_true_coeff))
#print('model: ' + str(test_model))
#print('responses: ' + str(test_responses))

#lasso_1 = LassoCV(fit_intercept = False).fit(test_model,test_responses[:,0])
#print(lasso_1.coef_)

#lasso_2 = LassoCV(fit_intercept = False).fit(test_model,test_responses[:,5])
#print(lasso_2.coef_)

#print(LSE(test_model,test_responses[:,0]))
#print(LSE(test_model,test_responses[:,5]))

#R_1,lse_1 = construct_R(test_model,test_responses[:,0])
#R_2,lse_2 = construct_R(test_model,test_responses[:,1])
#R_3,lse_3 = construct_R(test_model,test_responses[:,2])
#R_4,lse_4 = construct_R(test_model,test_responses[:,3])
#R_5,lse_5 = construct_R(test_model,test_responses[:,4])
#R_6,lse_6 = construct_R(test_model,test_responses[:,5])

#her_D_test = heredity_list(12,6)
#print(np.array([test_responses[:,0],test_responses[:,5]]))
#decomp = decomp_orthog(test_responses.T,[R_1,R_2,R_3,R_4,R_5,R_6],her_D_test,6,t=300,focus=0)

#print(decomp_scaled_coefficients(decomp[0].T,np.array([lse_1,lse_2,lse_3,lse_4,lse_5,lse_6]).T))
#print(decomp)

In [ ]:
rng = np.random.default_rng(100)
#Begin the experiment.
#For this experiment we have I=12 inputs and J=6 objective functions.

a_len = len(a)
decomp_lvl_len = len(decomp_lvl)
N_len = len(N)
#Create the strong heredity relations for a full RSM with 12 inputs. 
heredity = heredity_list(12,6)
for i in range(a_len):
    a_curr = a[i]
    for j in range(decomp_lvl_len):
        decomp_lvl_curr = decomp_lvl[j]
        #Create the true coefficient matrix and its corresponding indicator matrix.
        true_coeff = true_coefficient_matrix(decomp_lvl_curr,a_curr)
        true_indicator = active_indicator(true_coeff)
        #Create a dictionary to hold the two responses for the experiment
        resp_dict = { 'Replication': [],'N': [], 'Method': [], 'MSE': [], 'Correct Variable Selection': []}
        for k in range(outer_rep):
            #We create the whole response matrix with 800 responses.
            inputs = rng.uniform(0.0,1.0,size = (N[N_len-1],12))
            errors = rng.normal(0.0,var_error,size = (N[N_len-1],6))
            model_mat = two_way_interaction(inputs)
            responses = response_simulator(model_mat,true_coeff,errors)
            for n in range(N_len):
                print('i,j,k,n: ' + str(i) + ', ' + str(j) + ', ' + str(k) + ', ' + str(n))
                errors_n = errors[0:N[n],:]
                model_mat_n = model_mat[0:N[n],:]
                responses_n = responses[0:N[n],:]

                #Fit cross-validated LASSO and gather responses
                lasso_n = gather_lasso_reg(model_mat_n,responses_n)

                lasso_mse = global_mse(lasso_n,true_coeff)

                lasso_indicator = active_indicator(lasso_n)

                lasso_var_sel_percent = check_variable_selection(lasso_indicator,true_indicator)

                resp_dict['Replication'].append(k)
                resp_dict['N'].append(str(N[n]))
                resp_dict['Method'].append('LASSO')
                resp_dict['MSE'].append(lasso_mse)
                resp_dict['Correct Variable Selection'].append(lasso_var_sel_percent)

                #Fit decomposed-regression and gather responses
                R_1,lse_1 = construct_R(model_mat_n,responses_n[:,0])
                R_2,lse_2 = construct_R(model_mat_n,responses_n[:,1])
                R_3,lse_3 = construct_R(model_mat_n,responses_n[:,2])
                R_4,lse_4 = construct_R(model_mat_n,responses_n[:,3])
                R_5,lse_5 = construct_R(model_mat_n,responses_n[:,4])
                R_6,lse_6 = construct_R(model_mat_n,responses_n[:,5])
                
                decomp_fit_n = decomp_orthog(responses_n.T,[R_1,R_2,R_3,R_4,R_5,R_6],heredity,decomp_lvl_curr,t=300,focus=0)

                decomp_n = decomp_scaled_coefficients(decomp_fit_n[0].T,np.array([lse_1,lse_2,lse_3,lse_4,lse_5,lse_6]).T)

                decomp_mse = global_mse(decomp_n,true_coeff)

                decomp_indicator = active_indicator(decomp_n)

                decomp_var_sel_percent = check_variable_selection(decomp_indicator,true_indicator)

                resp_dict['Replication'].append(k)
                resp_dict['N'].append(str(N[n]))
                resp_dict['Method'].append('DECOMP')
                resp_dict['MSE'].append(decomp_mse)
                resp_dict['Correct Variable Selection'].append(decomp_var_sel_percent)

                print(resp_dict)

        df = pd.DataFrame(data = resp_dict)
        df.to_csv('consistency_experiment_a_'+str(a_curr)+'_decomp_lvl_'+str(decomp_lvl_curr)+'.csv')

        #MSE
        fig_1 = plt.figure()
        graph_1 = sns.boxplot(data = df, x = 'N', y = 'MSE', hue = 'Method')
        plt.title('MSE: a = ' +str(a_curr)+ ', Number of Clusters = ' + str(decomp_lvl_curr))
        fig_1.savefig('MSE_a_'+str(a_curr)+'_decomp_lvl_'+str(decomp_lvl_curr)+'.png',bbox_inches = 'tight')

        #Correct Variable Selection
        fig_2 = plt.figure()
        graph_2 = sns.boxplot(data = df, x = 'N', y = 'Correct Variable Selection', hue = 'Method')
        plt.title('CVS: a = ' +str(a_curr)+ ', Number of Clusters = ' + str(decomp_lvl_curr))
        fig_2.savefig('CVS_a_'+str(a_curr)+'_decomp_lvl_'+str(decomp_lvl_curr)+'.png',bbox_inches = 'tight')

In [ ]:
#Start experiment with perturbed coefficients
#parameter settings
eps = [0.1,0.25]
dens_spar = [0,1]
a_fix = 1.0
decomp_fix = 2

N = [200,400,800] #number of samples
N_len = len(N)
outer_rep = 20
var_error = 1.0 #variance of error terms.

#Create the strong heredity relations for a full RSM with 12 inputs. 
heredity = heredity_list(12,6)

rng = np.random.default_rng(100)#1000

for i in range(len(dens_spar)):
    dens_spar_curr = dens_spar[i]
    for j in range(len(eps)):
        eps_curr = eps[j]
        #Create the true coefficient matrix and its corresponding indicator matrix.
        true_coeff,pert_coeff = true_coefficient_matrix(decomp_fix,a_fix,dens_spar_curr,eps_curr)
        true_indicator = active_indicator(true_coeff)
        #Create a dictionary to hold the two responses for the experiment
        resp_dict = { 'Replication': [],'N': [], 'Method': [], 'MSE': [], 'Correct Variable Selection': []}

        for k in range(outer_rep):
            #We create the whole response matrix with 800 responses.
            inputs = rng.uniform(0.0,1.0,size = (N[N_len-1],12))
            errors = rng.normal(0.0,var_error,size = (N[N_len-1],6))
            model_mat = two_way_interaction(inputs)
            responses = response_simulator(model_mat,true_coeff+pert_coeff,errors)
            for n in range(N_len):
                print('i,j,k,n: ' + str(i) + ', ' + str(j) + ', ' + str(k) + ', ' + str(n))
                errors_n = errors[0:N[n],:]
                model_mat_n = model_mat[0:N[n],:]
                responses_n = responses[0:N[n],:]

                #Fit cross-validated LASSO and gather responses
                lasso_n = gather_lasso_reg(model_mat_n,responses_n)

                lasso_mse = global_mse(lasso_n,true_coeff)

                lasso_indicator = active_indicator(lasso_n)

                lasso_var_sel_percent = check_variable_selection(lasso_indicator,true_indicator)

                resp_dict['Replication'].append(k)
                resp_dict['N'].append(str(N[n]))
                resp_dict['Method'].append('LASSO')
                resp_dict['MSE'].append(lasso_mse)
                resp_dict['Correct Variable Selection'].append(lasso_var_sel_percent)

                #Fit decomposed-regression and gather responses
                R_1,lse_1 = construct_R(model_mat_n,responses_n[:,0])
                R_2,lse_2 = construct_R(model_mat_n,responses_n[:,1])
                R_3,lse_3 = construct_R(model_mat_n,responses_n[:,2])
                R_4,lse_4 = construct_R(model_mat_n,responses_n[:,3])
                R_5,lse_5 = construct_R(model_mat_n,responses_n[:,4])
                R_6,lse_6 = construct_R(model_mat_n,responses_n[:,5])
                
                decomp_fit_n = decomp_orthog(responses_n.T,[R_1,R_2,R_3,R_4,R_5,R_6],heredity,decomp_fix,t=300,focus=0)

                decomp_n = decomp_scaled_coefficients(decomp_fit_n[0].T,np.array([lse_1,lse_2,lse_3,lse_4,lse_5,lse_6]).T)

                decomp_mse = global_mse(decomp_n,true_coeff)

                decomp_indicator = active_indicator(decomp_n)

                decomp_var_sel_percent = check_variable_selection(decomp_indicator,true_indicator)

                resp_dict['Replication'].append(k)
                resp_dict['N'].append(str(N[n]))
                resp_dict['Method'].append('DECOMP')
                resp_dict['MSE'].append(decomp_mse)
                resp_dict['Correct Variable Selection'].append(decomp_var_sel_percent)

        df = pd.DataFrame(data = resp_dict)
        df.to_csv('consistency_experiment_dens_'+str(dens_spar_curr)+'_eps_'+str(eps_curr)+'.csv')

        if dens_spar_curr == 0:
            dens_str = 'Dense'
        else:
            dens_str = 'Sparse'
        #MSE
        fig_1 = plt.figure()
        graph_1 = sns.boxplot(data = df, x = 'N', y = 'MSE', hue = 'Method')
        plt.title('MSE: '+dens_str+', eps. = ' + str(eps_curr))
        fig_1.savefig('MSE_dens_'+str(dens_spar_curr)+'_eps_'+str(eps_curr)+'.png',bbox_inches = 'tight')

        #Correct Variable Selection
        fig_2 = plt.figure()
        graph_2 = sns.boxplot(data = df, x = 'N', y = 'Correct Variable Selection', hue = 'Method')
        plt.title('CVS: ' +dens_str+ ', eps. = ' + str(eps_curr))
        fig_2.savefig('CVS_dens_'+str(dens_spar_curr)+'_eps_'+str(eps_curr)+'.png',bbox_inches = 'tight')